# Exercises XP: Deep Learning Fundamentals

**Course:** Developers Institute  **Week 5 - Day 5**  
**Author:** Alex Goldbaum

Six exercises covering: deep-learning concepts (quiz), a manual perceptron, a
Keras MNIST classifier, manual forward propagation for a regression problem, a
from-scratch forward + backward pass in pure NumPy, and visualization of model
predictions on MNIST.


## Setup


In [ ]:
%pip install -qU tensorflow


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Exercise 1 — Small Quiz

**1. Key difference between traditional ML and deep learning.**
Traditional ML requires the engineer to hand-craft features that capture the
structure of the problem; the model then learns a fairly shallow mapping from
those features to the output. Deep learning replaces feature engineering with
**representation learning**: a network of many layers learns the features and
the mapping jointly, end-to-end, directly from raw inputs (pixels, text, audio).

**2. How do ANNs mimic the human brain?**
An artificial neuron loosely mimics a biological one: it sums weighted inputs,
passes the sum through a non-linear activation function, and fires an output to
the next layer. Stacking many neurons creates a network of connections where
each unit aggregates signals from many others, similar in spirit to dendrite →
soma → axon. The analogy is structural, not biological — real neurons fire spikes
and have far more complex dynamics.

**3. Why does deep learning perform better on large datasets?**
Deep networks have millions of parameters, which gives them enormous capacity to
fit complex functions. With a small dataset, that capacity leads to overfitting.
With millions of examples, the same capacity becomes an advantage: the model can
learn rich, hierarchical features (edges → textures → object parts → objects in
vision) that simpler models cannot represent.

**4. Challenges of deep learning and how to address them.**
- *Data hunger* → transfer learning, data augmentation, synthetic data, self-supervised pretraining.
- *Compute cost* → GPUs/TPUs, mixed precision, distillation, quantization.
- *Overfitting* → dropout, weight decay, batch normalization, early stopping.
- *Black-box interpretability* → saliency maps, SHAP, attention visualization, sparser architectures.
- *Brittleness to distribution shift* → robust training, domain adaptation, monitoring + retraining.

**5. What is feature engineering and why is it not needed in deep learning?**
Feature engineering is the manual process of deriving informative inputs from
raw data (e.g., turning a date into 'day-of-week', 'is-holiday'). Deep learning
models learn these representations on their own through their layers, so we feed
them raw or lightly preprocessed inputs and let backpropagation discover the
useful features.

**6. What role do hidden layers play in a DL model?**
Hidden layers progressively transform the input into more abstract
representations. Early layers detect low-level patterns (edges in images,
n-grams in text); deeper layers compose those into higher-level concepts
(faces, sentiment). The 'depth' of deep learning is literally the number of
hidden layers.

**7. Function of an activation function.**
An activation function introduces **non-linearity**. Without it, a stack of
linear layers would collapse into a single linear transformation and the
network could only learn linear relationships. ReLU, sigmoid, tanh, and
softmax are common choices — each non-linear, each shaping the output range
differently.


## Exercise 2 — Building a Simple Perceptron Decision System

Goal: decide whether to go outside based on `Temperature` (°F) and `Rain`
(1=yes, 0=no), using the formula

  Weighted Sum = (Temperature × 0.6) + (Rain × 0.4) + Bias

Bias = 2. Step activation: output 1 if Weighted Sum > 20, else 0.


In [ ]:
def perceptron_decision(temperature, rain, w_temp=0.6, w_rain=0.4, bias=2, threshold=20):
    weighted_sum = temperature * w_temp + rain * w_rain + bias
    output = 1 if weighted_sum > threshold else 0
    return weighted_sum, output

# Case 1: Temperature = 70F, Rain = 0
ws1, out1 = perceptron_decision(70, 0)
print(f'Case 1 (T=70F, Rain=No): weighted_sum = {ws1:.2f} -> output = {out1}',
      "-> GO OUTSIDE" if out1 == 1 else '-> STAY INSIDE')

# Case 2: Temperature = 50F, Rain = 1
ws2, out2 = perceptron_decision(50, 1)
print(f'Case 2 (T=50F, Rain=Yes): weighted_sum = {ws2:.2f} -> output = {out2}',
      '-> GO OUTSIDE' if out2 == 1 else '-> STAY INSIDE')


**Interpretation.**
- Case 1: weighted sum = 70·0.6 + 0·0.4 + 2 = **44**, which is > 20 → output **1** → go outside.
- Case 2: weighted sum = 50·0.6 + 1·0.4 + 2 = **32.4**, which is also > 20 → output **1** → go outside.

Both cases suggest going outside, but for different reasons. The perceptron
rewards temperature heavily (weight 0.6) and rain is barely a deterrent (weight
only 0.4, and rain is binary 0/1). With the given weights, even a 50°F rainy
day still clears the threshold. If we wanted rain to actually keep us inside
we would either increase its weight (e.g., to 20) **or** make it a negative
penalty (e.g., `−15 * rain`). This is exactly the kind of intuition that
gradient descent automates in real neural networks: it picks weights so that
the decision boundary matches the labels we provide.


## Exercise 3 — Simple Neural Network on MNIST

Architecture: Flatten → Dense(128, ReLU) → Dense(10, softmax). Loss
`categorical_crossentropy`, optimizer `adam`, metric `accuracy`.


In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.utils import to_categorical

tf.random.set_seed(RANDOM_STATE)

# 2. Load
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print('Train images:', x_train.shape, '| labels:', y_train.shape)
print('Test images :', x_test.shape, '| labels:', y_test.shape)


In [ ]:
# 3. Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 4. One-hot encode the labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)
print('One-hot example:', y_train[0], '->', y_train_cat[0])


In [ ]:
# 5. Build the model
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax'),
])
model.summary()


In [ ]:
# 6. Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

# 7. Train + evaluate
history = model.fit(
    x_train, y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1,
)

test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'\nTest loss    : {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


## Exercise 4 — Forward Propagation for House Price

Inputs x₁ = 2000 (sq ft), x₂ = 3 (bedrooms). Weights w₁ = 0.5, w₂ = 0.7.
Bias b = 50,000. Activation: ReLU.

  z = w₁·x₁ + w₂·x₂ + b
  ŷ = ReLU(z) = max(0, z)


In [ ]:
x1, x2 = 2000, 3
w1, w2 = 0.5, 0.7
b = 50_000

z = w1 * x1 + w2 * x2 + b
y_hat = max(0.0, z)  # ReLU

print(f'z = {w1}*{x1} + {w2}*{x2} + {b:,} = {z:,.2f}')
print(f'Predicted house price = ReLU(z) = ${y_hat:,.2f}')


**Interpretation.** The pre-activation `z = 51,002.10` is already positive, so
ReLU passes it through unchanged. The model predicts a house price of about
**$51,002**. The bias (50,000) sets a base price and the weighted contributions
of square footage ($0.50/sq ft) and bedrooms ($0.70/bedroom) add the small
increment. These weights are clearly toy values — in a real regression we would
fit them from data, and we would also expect ReLU to behave linearly here
because house prices are virtually always positive.


## Exercise 5 — Forward + Backward Propagation in Python

Toy regression: predict an exam score from hours studied and the previous
test score. We implement one full step of gradient descent by hand: forward
pass, MSE loss, analytical gradients, weight + bias update.


In [ ]:
# Initialize input data (features)
x = np.array([4, 80])  # 4 hours studied, previous test score: 80

# Initialize weights and bias
w = np.array([0.6, 0.3])  # Initial weights
b = 10                    # Initial bias


def forward_propagation(x, w, b):
    z = np.dot(x, w) + b  # Weighted sum
    return z              # Linear activation (regression)


# Compute prediction
y_pred = forward_propagation(x, w, b)
y_true = 85

# Mean Squared Error (1/2 factor cancels the 2 from the derivative)
loss = 0.5 * (y_true - y_pred) ** 2

# Gradients (chain rule on the MSE)
grad_w = -(y_true - y_pred) * x
grad_b = -(y_true - y_pred)

# Update weights and bias
learning_rate = 0.01
w_new = w - learning_rate * grad_w
b_new = b - learning_rate * grad_b

print(f'Initial Prediction: {y_pred}')
print(f'Loss              : {loss}')
print(f'Updated Weights   : {w_new}')
print(f'Updated Bias      : {b_new}')


**Why does gradient descent reduce the error?**
The gradient points in the direction of *steepest increase* of the loss with
respect to each parameter. Subtracting the gradient (times a small learning
rate) moves the parameters in the direction of *steepest decrease* — so by
definition the loss goes down (for a small enough step). Repeating this
thousands of times converges toward a local minimum of the loss.

**Quick experiment — multiple gradient-descent steps.**


In [ ]:
# Run the same step in a loop and watch the loss collapse.
# The inputs have unequal magnitudes (4 vs 80), so we use a small learning rate
# to keep the updates stable (a larger lr would overshoot and diverge).
w_loop = np.array([0.6, 0.3], dtype=float)
b_loop = 10.0
lr = 1e-4

history_loss = []
for step in range(200):
    y_p = forward_propagation(x, w_loop, b_loop)
    err = y_true - y_p
    history_loss.append(0.5 * err ** 2)
    w_loop = w_loop + lr * err * x
    b_loop = b_loop + lr * err

print(f'Final w: {w_loop} | final b: {b_loop:.4f} | final loss: {history_loss[-1]:.6f}')

plt.figure(figsize=(8, 4))
plt.plot(history_loss, marker='o', markersize=3)
plt.xlabel('Gradient-descent step')
plt.ylabel('MSE loss')
plt.title('Loss decreasing across gradient-descent steps', fontweight='bold')
plt.yscale('log')
plt.tight_layout()
plt.show()


**What happens if we change the learning rate?**
- Too small (e.g., `1e-5`): training takes forever; the loss drops imperceptibly per step.
- Too large (e.g., `0.1`): the updates overshoot the minimum and the loss
  *grows* exponentially. You can verify this by replacing `lr = 0.001` with `lr = 0.1` in the loop above.
- A reasonable middle ground (1e-3 here) gives smooth, monotonic convergence.


## Exercise 6 — Visualize MNIST Predictions

We reuse the model trained in Exercise 3 (so we don't pay the training cost
twice) and visualize predictions on a small batch of test images.


In [ ]:
# Make predictions
preds = model.predict(x_test, verbose=0)
pred_labels = preds.argmax(axis=1)
true_labels = y_test  # integer labels (we have y_test_cat for the one-hot version)

# Pick a mix of correct + incorrect predictions so the grid is informative
rng = np.random.default_rng(RANDOM_STATE)
correct_idx = np.where(pred_labels == true_labels)[0]
wrong_idx = np.where(pred_labels != true_labels)[0]

sample_correct = rng.choice(correct_idx, size=12, replace=False)
sample_wrong = rng.choice(wrong_idx, size=4, replace=False) if len(wrong_idx) >= 4 else wrong_idx
sample_idx = np.concatenate([sample_correct, sample_wrong])

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, i in zip(axes.flat, sample_idx):
    ax.imshow(x_test[i], cmap='gray')
    is_correct = pred_labels[i] == true_labels[i]
    color = 'seagreen' if is_correct else 'tomato'
    ax.set_title(f'Pred: {pred_labels[i]} (True: {true_labels[i]})', color=color, fontsize=10)
    ax.axis('off')

plt.suptitle('MNIST predictions — green = correct, red = wrong',
             y=1.0, fontweight='bold')
plt.tight_layout()
plt.show()


## Conclusions

- The **perceptron** showed that weights and bias define a decision boundary,
  and that picking weights manually is hard — gradient descent automates this.
- The **MNIST Keras model** reaches ~97-98% test accuracy with just one hidden
  layer of 128 neurons after a handful of epochs. Deep learning's strength
  shows up here without needing convolutions yet.
- The **manual forward propagation** for house prices makes the math behind a
  neural network explicit: a weighted sum, a bias, and an activation.
- The **backward propagation** demo shows the engine that makes all of this
  work: gradient descent updates weights to reduce the loss step by step.
- The **prediction visualizations** make model behaviour tangible — most
  digits are easy, but the model still struggles with ambiguous handwriting.
